## Q6.
UCI-HAR dataset. Compare DT, RF and Linear regression (yes, regression). For linear regression: each class label as an integer value. Say, 1: Sitting, 2:..., and so on. Use features extracted (from flattened out Linear Acceleration) using the TSFEL library. Compare the performance of these models. Is the usage of linear regression for classification justified? Why or why not? [2 Marks]

In [ ]:
# importing the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression

1) Load the data

In [ ]:
X = pd.read_csv('X_Tsfel_data.csv') #reading the data
Y = pd.read_csv('y_data.csv') #reading the labels
classes = {1:"Walking",2:"WalkingUp",3:"WalkingDown",4:"Sitting",5:"Standing",6:"Laying"} #creating a dictionary of the labels
features = X.columns #getting the column names

In [ ]:
X.shape #getting the shape of the data

In [ ]:
Y #displaying the labels

2) Performing train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42) #splitting the data into training and testing data


In [ ]:
y_test = y_test.values.ravel() #reshaping the data
y_train = y_train.values.ravel() #reshaping the data


3) Feature Scaling using MinMaxScaler

In [ ]:
#Minmax scaler
scaler = MinMaxScaler() #creating a minmax scaler
X_train = scaler.fit_transform(X_train) #fitting the scaler to the training data
X_test = scaler.transform(X_test) #transforming the testing data


4.1 Fitting a Decision Tree Model


In [ ]:
# Finding the better criterion by validation
criteria = ['gini', 'entropy'] # Creating an array for storing the criteria
gini_dict = {} # Creating a dictionary for storing the accuracy of the model for each max depth for criterion = gini
entropy_dict = {} # Creating a dictionary for storing the accuracy of the model for each max depth for criterion = entropy
for criterion in criteria: # Looping over the criteria
    for i in range(2,11): # Looping over the max depth
        dtree = DecisionTreeClassifier(max_depth=i, criterion=criterion, random_state=14) # Creating a decision tree classifier of max depth = i and criterion = criterion
        arr = cross_val_score(dtree, X_train, y_train, cv=10) # Calculating the cross validation score
        if criterion == 'gini': # Checking the criterion
            gini_dict[i] = arr.mean() # Storing the accuracy of the model for each max depth for criterion = gini
        else:
            entropy_dict[i] = arr.mean() # Storing the accuracy of the model for each max depth for criterion = entropy

plt.figure(figsize=(15, 5))
plt.plot(gini_dict.keys(), gini_dict.values(), label='gini') # Plotting the accuracy of the model for each max depth for criterion = gini
plt.plot(entropy_dict.keys(), entropy_dict.values(), label='entropy') # Plotting the accuracy of the model for each max depth for criterion = entropy
plt.xlabel("Height of the Tree")
plt.ylabel("Accuracy in %")
plt.title("The Mean Cross Validation Accuracy against the height of the tree for different criteria")
plt.legend()

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=4, criterion= "entropy", random_state=14) # Creating a decision tree classifier of max depth = 4 and criterion = entropy
dt_model.fit(X_train, y_train) # Fitting the model to the training data
y_pred_dt = dt_model.predict(X_test) # Predicting the labels of the testing data
print("Accuracy:", accuracy_score(y_test, y_pred_dt)) # Printing the accuracy of the model
print(classification_report(y_test, y_pred_dt))
import seaborn as sns
cm = confusion_matrix(y_test, y_pred_dt) # Creating the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=classes.values(), yticklabels=classes.values())
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion Matrix for Decision Tree Classifier')
plt.show()

In [ ]:
y_pred_dt

4.2 Random Forest

In [ ]:
rf_dict = {}
for n_estimators in range(1, 200):
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=14) # Creating a random forest classifier of n_estimators = n_estimators
    arr = cross_val_score(rf, X_train, y_train, cv=10) # Calculating the cross validation score
    rf_dict[n_estimators] = arr.mean() # Storing the accuracy of the model for each n_estimators
    print("The accuracy of the model for n_estimators = ", n_estimators, " is ", arr.mean())

plt.figure(figsize=(15, 5))
plt.plot(rf_dict.keys(), rf_dict.values(), label='Random Forest') # Plotting the accuracy of the model for each n_estimators
plt.xlabel("Number of Trees")
plt.ylabel("Accuracy in %")
plt.title("The Mean Cross Validation Accuracy against the number of trees for Random Forest")
plt.legend()


In [ ]:
# random forest
rf_model = RandomForestClassifier(n_estimators=9, random_state=14) # Creating a random forest classifier of n_estimators = 100 
rf_model.fit(X_train, y_train) # Fitting the model to the training data
y_pred_rf = rf_model.predict(X_test) # Predicting the labels of the testing data
print("Accuracy:", accuracy_score(y_test, y_pred_rf)) # Printing the accuracy of the model
print(classification_report(y_test, y_pred_rf)) # Printing the classification report
cm = confusion_matrix(y_test, y_pred_rf) # Creating the confusion matrix
plt.figure(figsize=(10, 7)) # Plotting the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=classes.values(), yticklabels=classes.values()) # Plotting the confusion matrix
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion Matrix for Random Forest Classifier')
plt.show()


In [ ]:
y_pred_rf #displaying the predicted labels

4.3 Linear Regression

In [ ]:
lr_model = LinearRegression() # Creating a linear regression model
lr_model.fit(X_train, y_train) # Fitting the model to the training data
y_pred_lr = lr_model.predict(X_test) # Predicting the labels of the testing data
y_pred_round = np.round(y_pred_lr) # Rounding the predicted labels



In [ ]:
y_pred_lr #displaying the predicted labels

In [ ]:
# apply clip to round the values
y_pred_round = np.clip(y_pred_round, 1, 6)

In [ ]:
y_pred_round #displaying the predicted labels

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_round) # Creating the confusion matrix
plt.figure(figsize=(10, 7)) # Plotting the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=classes.values(), yticklabels=classes.values()) # Plotting the confusion matrix
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion Matrix for Linear Regression')
plt.show()

In [ ]:
# Classification report
print("Accuracy:", accuracy_score(y_test, y_pred_round)) # Printing the accuracy of the model
print(classification_report(y_test, y_pred_round)) # Printing the classification report

Using Minimax scaler function

In [ ]:
scaler = MinMaxScaler(feature_range=(1, 6))
normalized_arr = scaler.fit_transform(y_pred_lr.reshape(-1, 1)).flatten()
normalized_arr


In [ ]:
y_pred_round = np.round(normalized_arr) # Rounding the predicted labels
y_pred_round #displaying the predicted labels


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_round)) # Printing the accuracy of the model\
print(classification_report(y_test, y_pred_round)) # Printing the classification report
cm = confusion_matrix(y_test, y_pred_round) # Creating the confusion matrix
plt.figure(figsize=(10, 7)) # Plotting the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=classes.values(), yticklabels=classes.values()) # Plotting the confusion matrix
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion Matrix for Linear Regression')
plt.show()


Using Sigmoid function

In [ ]:
# Using Sigmoid function to trim the values to range 1 to 6
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

y_pred_sig = sigmoid(y_pred_lr/6) # Applying the sigmoid function to the predicted labels
y_pred_sig = np.round(y_pred_sig*6) # Rounding the predicted labels
print(y_pred_sig) #displaying the predicted labels

In [ ]:
print(classification_report(y_test, y_pred_sig)) # Printing the classification report
print("Accuracy:", accuracy_score(y_test, y_pred_sig)) # Printing the accuracy of the model
cm = confusion_matrix(y_test, y_pred_sig) # Creating the confusion matrix
plt.figure(figsize=(10, 7)) # Plotting the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=classes.values(), yticklabels=classes.values()) # Plotting the confusion matrix
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion Matrix for Linear Regression')
plt.show()

## Q7
Obtain the weights (take absolute values as weights can also be negative) of the linear regression model. Also, obtain the feature importance from the Random Forest model. Plot the weights obtained as a Bar plot. This will help you visualize what features are being prioritized by the models. Note that sum of feature importances for a Random Forest model is 1. you will have to bring the linear regression weights to the same scale. To do so you can divide the weights by the sum of all the weights. Plot the importance of the features in the same plot. Figure out the top 10 important features obtained from both the models and display their names. What do you infer? [1 Mark]

Feature importance from random forest

In [ ]:
feature_importances_rf = rf_model.feature_importances_
print(feature_importances_rf)

In [ ]:
# feature importance from random forest
feature_importances_rf = rf_model.feature_importances_
sum = np.sum(feature_importances_rf)
sum

linear regression weights(absolute value)

In [ ]:
parameter = lr_model.coef_
print(parameter)

In [ ]:
#linear regression weights
parameter = lr_model.coef_
parameter = np.abs(parameter)
parameter = parameter / np.sum(parameter)
sum = np.sum(parameter)
sum

Plotting



In [ ]:
# plotiing the feature importance and parameter for earch feature

# plot the feature importance
plt.figure(figsize=(15, 50))
plt.barh(features, feature_importances_rf)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Feature Importance for Random Forest')
plt.show()

# plot the parameter
plt.figure(figsize=(15, 50))
plt.barh(features, parameter)
plt.xlabel('Parameter')
plt.ylabel('Features')
plt.title('Parameter for Linear Regression')
plt.show()



In [ ]:
# plot the feature importance and parameter for each feature
plt.figure(figsize=(70, 30))
plt.bar(np.array(list(map(int, list(features)))) - 0.25, parameter, 0.5, label = 'Linear Regression Parameters') 
plt.bar(np.array(list(map(int, list(features)))) + 0.25, feature_importances_rf, 0.5, label = 'Random Forest Feature Importance') 

TOP 10

In [ ]:
# get top 10 features
feature_importances = pd.DataFrame(rf_model.feature_importances_,
                                    columns=['importance']).sort_values('importance', ascending=False)
sum = np.sum(feature_importances)
print(sum)
fitop10 = pd.DataFrame({"Features": feature_importances.head(10).index, "Feature Importance": feature_importances.head(10).values.flatten()})
fitop10

In [ ]:
# top 10 weights from linear regression
parameter = pd.DataFrame(lr_model.coef_, columns=['parameter']).sort_values('parameter', ascending=False)
sum = np.sum(parameter)
print(sum)
paratop10 = pd.DataFrame({"Features": parameter.head(10).index, "Parameter Value": parameter.head(10).values.flatten()})
paratop10

In [ ]:
ans = ['0_Absolute energy', '0_Area under the curve', '0_Autocorrelation', '0_Average power', '0_Centroid', '0_ECDF Percentile Count_0', '0_ECDF Percentile Count_1', '0_ECDF Percentile_0', '0_ECDF Percentile_1', '0_ECDF_0', '0_ECDF_1', '0_ECDF_2', '0_ECDF_3', '0_ECDF_4', '0_ECDF_5', '0_ECDF_6', '0_ECDF_7', '0_ECDF_8', '0_ECDF_9', '0_Entropy', '0_FFT mean coefficient_0', '0_FFT mean coefficient_1', '0_FFT mean coefficient_10', '0_FFT mean coefficient_100', '0_FFT mean coefficient_101', '0_FFT mean coefficient_102', '0_FFT mean coefficient_103', '0_FFT mean coefficient_104', '0_FFT mean coefficient_105', '0_FFT mean coefficient_106', '0_FFT mean coefficient_107', '0_FFT mean coefficient_108', '0_FFT mean coefficient_109', '0_FFT mean coefficient_11', '0_FFT mean coefficient_110', '0_FFT mean coefficient_111', '0_FFT mean coefficient_112', '0_FFT mean coefficient_113', '0_FFT mean coefficient_114', '0_FFT mean coefficient_115', '0_FFT mean coefficient_116', '0_FFT mean coefficient_117', '0_FFT mean coefficient_118', '0_FFT mean coefficient_119', '0_FFT mean coefficient_12', '0_FFT mean coefficient_120', '0_FFT mean coefficient_121', '0_FFT mean coefficient_122', '0_FFT mean coefficient_123', '0_FFT mean coefficient_124', '0_FFT mean coefficient_125', '0_FFT mean coefficient_126', '0_FFT mean coefficient_127', '0_FFT mean coefficient_128', '0_FFT mean coefficient_129', '0_FFT mean coefficient_13', '0_FFT mean coefficient_130', '0_FFT mean coefficient_131', '0_FFT mean coefficient_132', '0_FFT mean coefficient_133', '0_FFT mean coefficient_134', '0_FFT mean coefficient_135', '0_FFT mean coefficient_136', '0_FFT mean coefficient_137', '0_FFT mean coefficient_138', '0_FFT mean coefficient_139', '0_FFT mean coefficient_14', '0_FFT mean coefficient_140', '0_FFT mean coefficient_141', '0_FFT mean coefficient_142', '0_FFT mean coefficient_143', '0_FFT mean coefficient_144', '0_FFT mean coefficient_145', '0_FFT mean coefficient_146', '0_FFT mean coefficient_147', '0_FFT mean coefficient_148', '0_FFT mean coefficient_149', '0_FFT mean coefficient_15', '0_FFT mean coefficient_150', '0_FFT mean coefficient_151', '0_FFT mean coefficient_152', '0_FFT mean coefficient_153', '0_FFT mean coefficient_154', '0_FFT mean coefficient_155', '0_FFT mean coefficient_156', '0_FFT mean coefficient_157', '0_FFT mean coefficient_158', '0_FFT mean coefficient_159', '0_FFT mean coefficient_16', '0_FFT mean coefficient_160', '0_FFT mean coefficient_161', '0_FFT mean coefficient_162', '0_FFT mean coefficient_163', '0_FFT mean coefficient_164', '0_FFT mean coefficient_165', '0_FFT mean coefficient_166', '0_FFT mean coefficient_167', '0_FFT mean coefficient_168', '0_FFT mean coefficient_169', '0_FFT mean coefficient_17', '0_FFT mean coefficient_170', '0_FFT mean coefficient_171', '0_FFT mean coefficient_172', '0_FFT mean coefficient_173', '0_FFT mean coefficient_174', '0_FFT mean coefficient_175', '0_FFT mean coefficient_176', '0_FFT mean coefficient_177', '0_FFT mean coefficient_178', '0_FFT mean coefficient_179', '0_FFT mean coefficient_18', '0_FFT mean coefficient_180', '0_FFT mean coefficient_181', '0_FFT mean coefficient_182', '0_FFT mean coefficient_183', '0_FFT mean coefficient_184', '0_FFT mean coefficient_185', '0_FFT mean coefficient_186', '0_FFT mean coefficient_187', '0_FFT mean coefficient_188', '0_FFT mean coefficient_189', '0_FFT mean coefficient_19', '0_FFT mean coefficient_190', '0_FFT mean coefficient_191', '0_FFT mean coefficient_192', '0_FFT mean coefficient_193', '0_FFT mean coefficient_194', '0_FFT mean coefficient_195', '0_FFT mean coefficient_196', '0_FFT mean coefficient_197', '0_FFT mean coefficient_198', '0_FFT mean coefficient_199', '0_FFT mean coefficient_2', '0_FFT mean coefficient_20', '0_FFT mean coefficient_200', '0_FFT mean coefficient_201', '0_FFT mean coefficient_202', '0_FFT mean coefficient_203', '0_FFT mean coefficient_204', '0_FFT mean coefficient_205', '0_FFT mean coefficient_206', '0_FFT mean coefficient_207', '0_FFT mean coefficient_208', '0_FFT mean coefficient_209', '0_FFT mean coefficient_21', '0_FFT mean coefficient_210', '0_FFT mean coefficient_211', '0_FFT mean coefficient_212', '0_FFT mean coefficient_213', '0_FFT mean coefficient_214', '0_FFT mean coefficient_215', '0_FFT mean coefficient_216', '0_FFT mean coefficient_217', '0_FFT mean coefficient_218', '0_FFT mean coefficient_219', '0_FFT mean coefficient_22', '0_FFT mean coefficient_220', '0_FFT mean coefficient_221', '0_FFT mean coefficient_222', '0_FFT mean coefficient_223', '0_FFT mean coefficient_224', '0_FFT mean coefficient_225', '0_FFT mean coefficient_226', '0_FFT mean coefficient_227', '0_FFT mean coefficient_228', '0_FFT mean coefficient_229', '0_FFT mean coefficient_23', '0_FFT mean coefficient_230', '0_FFT mean coefficient_231', '0_FFT mean coefficient_232', '0_FFT mean coefficient_233', '0_FFT mean coefficient_234', '0_FFT mean coefficient_235', '0_FFT mean coefficient_236', '0_FFT mean coefficient_237', '0_FFT mean coefficient_238', '0_FFT mean coefficient_239', '0_FFT mean coefficient_24', '0_FFT mean coefficient_240', '0_FFT mean coefficient_241', '0_FFT mean coefficient_242', '0_FFT mean coefficient_243', '0_FFT mean coefficient_244', '0_FFT mean coefficient_245', '0_FFT mean coefficient_246', '0_FFT mean coefficient_247', '0_FFT mean coefficient_248', '0_FFT mean coefficient_249', '0_FFT mean coefficient_25', '0_FFT mean coefficient_250', '0_FFT mean coefficient_26', '0_FFT mean coefficient_27', '0_FFT mean coefficient_28', '0_FFT mean coefficient_29', '0_FFT mean coefficient_3', '0_FFT mean coefficient_30', '0_FFT mean coefficient_31', '0_FFT mean coefficient_32', '0_FFT mean coefficient_33', '0_FFT mean coefficient_34', '0_FFT mean coefficient_35', '0_FFT mean coefficient_36', '0_FFT mean coefficient_37', '0_FFT mean coefficient_38', '0_FFT mean coefficient_39', '0_FFT mean coefficient_4', '0_FFT mean coefficient_40', '0_FFT mean coefficient_41', '0_FFT mean coefficient_42', '0_FFT mean coefficient_43', '0_FFT mean coefficient_44', '0_FFT mean coefficient_45', '0_FFT mean coefficient_46', '0_FFT mean coefficient_47', '0_FFT mean coefficient_48', '0_FFT mean coefficient_49', '0_FFT mean coefficient_5', '0_FFT mean coefficient_50', '0_FFT mean coefficient_51', '0_FFT mean coefficient_52', '0_FFT mean coefficient_53', '0_FFT mean coefficient_54', '0_FFT mean coefficient_55', '0_FFT mean coefficient_56', '0_FFT mean coefficient_57', '0_FFT mean coefficient_58', '0_FFT mean coefficient_59', '0_FFT mean coefficient_6', '0_FFT mean coefficient_60', '0_FFT mean coefficient_61', '0_FFT mean coefficient_62', '0_FFT mean coefficient_63', '0_FFT mean coefficient_64', '0_FFT mean coefficient_65', '0_FFT mean coefficient_66', '0_FFT mean coefficient_67', '0_FFT mean coefficient_68', '0_FFT mean coefficient_69', '0_FFT mean coefficient_7', '0_FFT mean coefficient_70', '0_FFT mean coefficient_71', '0_FFT mean coefficient_72', '0_FFT mean coefficient_73', '0_FFT mean coefficient_74', '0_FFT mean coefficient_75', '0_FFT mean coefficient_76', '0_FFT mean coefficient_77', '0_FFT mean coefficient_78', '0_FFT mean coefficient_79', '0_FFT mean coefficient_8', '0_FFT mean coefficient_80', '0_FFT mean coefficient_81', '0_FFT mean coefficient_82', '0_FFT mean coefficient_83', '0_FFT mean coefficient_84', '0_FFT mean coefficient_85', '0_FFT mean coefficient_86', '0_FFT mean coefficient_87', '0_FFT mean coefficient_88', '0_FFT mean coefficient_89', '0_FFT mean coefficient_9', '0_FFT mean coefficient_90', '0_FFT mean coefficient_91', '0_FFT mean coefficient_92', '0_FFT mean coefficient_93', '0_FFT mean coefficient_94', '0_FFT mean coefficient_95', '0_FFT mean coefficient_96', '0_FFT mean coefficient_97', '0_FFT mean coefficient_98', '0_FFT mean coefficient_99', '0_Fundamental frequency', '0_Histogram_0', '0_Histogram_1', '0_Histogram_2', '0_Histogram_3', '0_Histogram_4', '0_Histogram_5', '0_Histogram_6', '0_Histogram_7', '0_Histogram_8', '0_Histogram_9', '0_Human range energy', '0_Interquartile range', '0_Kurtosis', '0_LPCC_0', '0_LPCC_1', '0_LPCC_10', '0_LPCC_11', '0_LPCC_2', '0_LPCC_3', '0_LPCC_4', '0_LPCC_5', '0_LPCC_6', '0_LPCC_7', '0_LPCC_8', '0_LPCC_9', '0_MFCC_0', '0_MFCC_1', '0_MFCC_10', '0_MFCC_11', '0_MFCC_2', '0_MFCC_3', '0_MFCC_4', '0_MFCC_5', '0_MFCC_6', '0_MFCC_7', '0_MFCC_8', '0_MFCC_9', '0_Max', '0_Max power spectrum', '0_Maximum frequency', '0_Mean', '0_Mean absolute deviation', '0_Mean absolute diff', '0_Mean diff', '0_Median', '0_Median absolute deviation', '0_Median absolute diff', '0_Median diff', '0_Median frequency', '0_Min', '0_Negative turning points', '0_Neighbourhood peaks', '0_Peak to peak distance', '0_Positive turning points', '0_Power bandwidth', '0_Root mean square', '0_Signal distance', '0_Skewness', '0_Slope', '0_Spectral centroid', '0_Spectral decrease', '0_Spectral distance', '0_Spectral entropy', '0_Spectral kurtosis', '0_Spectral positive turning points', '0_Spectral roll-off', '0_Spectral roll-on', '0_Spectral skewness', '0_Spectral slope', '0_Spectral spread', '0_Spectral variation', '0_Standard deviation', '0_Sum absolute diff', '0_Variance', '0_Wavelet absolute mean_0', '0_Wavelet absolute mean_1', '0_Wavelet absolute mean_2', '0_Wavelet absolute mean_3', '0_Wavelet absolute mean_4', '0_Wavelet absolute mean_5', '0_Wavelet absolute mean_6', '0_Wavelet absolute mean_7', '0_Wavelet absolute mean_8', '0_Wavelet energy_0', '0_Wavelet energy_1', '0_Wavelet energy_2', '0_Wavelet energy_3', '0_Wavelet energy_4', '0_Wavelet energy_5', '0_Wavelet energy_6', '0_Wavelet energy_7', '0_Wavelet energy_8', '0_Wavelet entropy', '0_Wavelet standard deviation_0', '0_Wavelet standard deviation_1', '0_Wavelet standard deviation_2', '0_Wavelet standard deviation_3', '0_Wavelet standard deviation_4', '0_Wavelet standard deviation_5', '0_Wavelet standard deviation_6', '0_Wavelet standard deviation_7', '0_Wavelet standard deviation_8', '0_Wavelet variance_0', '0_Wavelet variance_1', '0_Wavelet variance_2', '0_Wavelet variance_3', '0_Wavelet variance_4', '0_Wavelet variance_5', '0_Wavelet variance_6', '0_Wavelet variance_7', '0_Wavelet variance_8', '0_Zero crossing rate']

Features_imp = [373,356,362,357,349,350,45,247,226,62]
paramVals = [56,114,39,237,215,197,73,65,105,68]

print("Top 10 features from Random Forest")

for i in range(10):
    print("Feature at rank", i+1, "is", ans[Features_imp[i]])


In [ ]:
print("Top 10 features from Linear Regression")

for i in range(10):
    print("Parameter at rank", i+1, "is", ans[paramVals[i]])